# Pipeline

One place to run the whole project: configure, collect, inspect, train, probe. Each section is a thin wrapper over code that lives in its own module, so this notebook is orchestration only.

Paths and the active run come from `config.py`. Change `RUN` there to point everything at a different dataset and checkpoint. Hyperparameters are set in the Config cell below and threaded through the rest.

## Config

Paths come from `config.py`. The knobs below are experiment choices for this run. `SEED` is shared by the data split and the probe so the probe always scores on truly held-out episodes.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "constants.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from config import DATA_PATH, CKPT_PATH, SEED

# --- Data generation (only used if DATA_PATH does not exist yet) ---
N_EPISODES = 5000
MAX_STEPS  = 100
HOLD_K     = 4
GRID_SIZE  = 40

# Action-sampling policy. Mirrors constants.py defaults; edit to sweep the data
# policy. Whatever is used here gets stored in the dataset's HDF5 attrs, so each
# file stays self-describing.
V_MEAN, V_STD         = 0.18, 0.05
OMEGA_MEAN, OMEGA_STD = 0.0, 0.6

# --- Model + training ---
LATENT_DIM = 128
LR         = 1e-3
LAM        = 0.01
EPOCHS     = 5
BATCH_SIZE = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"dataset    : {DATA_PATH}")
print(f"checkpoint : {CKPT_PATH}")
print(f"device     : {DEVICE}")

## Collect

Generate the dataset if it does not already exist. Skips straight through when `DATA_PATH` is present so re-running the notebook is cheap. Coverage diagnostics are written next to the dataset.

In [ ]:
from data.collect import collect_dataset
from data.coverage import plot_coverage

if DATA_PATH.exists():
    print(f"dataset exists, skipping collection: {DATA_PATH}")
else:
    collect_dataset(DATA_PATH, n_episodes=N_EPISODES, max_steps=MAX_STEPS,
                    hold_k=HOLD_K, grid_size=GRID_SIZE,
                    v_mean=V_MEAN, v_std=V_STD,
                    omega_mean=OMEGA_MEAN, omega_std=OMEGA_STD,
                    seed=SEED)
    plot_coverage(DATA_PATH, show=True)

## Inspect

Read-back before training. Reports dataset size plus the three diagnostics from `data/diagnostics.py`:

- **frame identity rate**: fraction of consecutive frames that are pixel-identical. Low is good. High means transitions are too redundant to learn from.
- **pixel-change distribution**: how many cells flip per step.
- **theta quantization ceiling**: the smallest rotation that changes a pixel at each resolution. The hard observability floor for heading.

In [ ]:
import matplotlib.pyplot as plt
from data.diagnostics import (
    load_h5,
    frame_identity_rate,
    pixel_change_distribution,
    theta_quantization_ceiling,
)

d = load_h5(DATA_PATH)
print(f"transitions : {d['actions'].shape[0]}")
print(f"episodes    : {len(d['episode_lengths'])}")
print(f"grid size   : {int(d['attrs']['grid_size'])}")

# fraction of consecutive frames that are pixel-identical (low is good)
rate = frame_identity_rate(d["frames"], d["episode_starts"], d["episode_lengths"])
print(f"frame identity rate : {rate:.2%}")

# per-step pixel-change distribution
_, ax = plt.subplots()
pixel_change_distribution(d["frames"], d["episode_starts"], d["episode_lengths"], ax=ax)
plt.show()

# heading observability floor per resolution
print("\ntheta quantization ceiling (smallest visible rotation):")
theta_quantization_ceiling()

## Train

Build the JEPA and train it, saving the best-val checkpoint to `CKPT_PATH`. `torch.manual_seed(SEED)` makes weight init reproducible. The collapse monitor (right panel of the curve) should hold near 1.

In [ ]:
from models.jepa import JEPA
from models.dataset import make_dataloaders
from models.train import train_jepa, plot_history

torch.manual_seed(SEED)
train_dl, val_dl = make_dataloaders(DATA_PATH, batch_size=BATCH_SIZE, seed=SEED)
model = JEPA(grid_size=GRID_SIZE, latent_dim=LATENT_DIM)

history = train_jepa(model, train_dl, val_dl, epochs=EPOCHS, lr=LR, lam=LAM,
                     device=DEVICE, save_best_to=CKPT_PATH)
plot_history(history)

## Probe

Freeze the trained encoder and fit linear + MLP probes to read `(x, y, theta)` out of the latent. Compare against the chance baseline: a probe that beats chance means the encoder learned physical state from pixels alone. Same `SEED` as training, so the probe's val episodes are the held-out ones.

In [ ]:
from eval.probe import run_probe

results = run_probe(ckpt_path=CKPT_PATH, latent_dim=LATENT_DIM,
                    batch_size=256, device=DEVICE, seed=SEED)